# Corretude de algoritmos iterativos e invariantes de laço — Parte 1 — Tutorial

**Algoritmos e Estruturas de Dados II (COMP0498) — UFS — 2026.2**

## Objetivos

Ao final deste tutorial você será capaz de:

- Enunciar o invariante de um laço e verificar as três propriedades: **inicialização**, **manutenção** e **término**;
- Instrumentar um programa em C para **observar** o invariante valendo a cada iteração;
- Provar a corretude da **busca do máximo** e usá-la como molde para laços de acumulação;
- Provar a corretude da **ordenação por inserção**, incluindo a cláusula de permutação;
- Usar o invariante para **diagnosticar** laços errados antes de qualquer teste.


In [ ]:
# Verifique se o gcc está disponível no seu ambiente
!gcc --version | head -1


## 1. O invariante como afirmação verificável

Um **invariante de laço** é uma afirmação sobre as variáveis do programa que é verdadeira
**no início de cada iteração**. A prova de corretude tem três passos: mostrar que o invariante
vale antes da primeira iteração (**inicialização**), que cada iteração o preserva
(**manutenção**) e que, ao sair do laço, ele implica o resultado desejado (**término**).

A prova é feita no papel — mas o computador pode nos ajudar a *conferir* o invariante em
execuções concretas usando `assert`. Se o `assert` falha, o invariante que enunciamos está
errado (ou o código está). Se passa, ganhamos confiança de que vale a pena prová-lo.


In [ ]:
%%writefile invariante_observado.c
#include <stdio.h>
#include <assert.h>

/* Soma de A[0..n-1], com o invariante conferido a cada iteracao */
int soma(int A[], int n) {
    int s = 0;
    for (int i = 0; i < n; i++) {
        /* invariante: s == A[0] + A[1] + ... + A[i-1] */
        int esperado = 0;                 /* recalcula a soma do prefixo */
        for (int k = 0; k < i; k++)
            esperado += A[k];
        assert(s == esperado);            /* confere o invariante        */
        printf("inicio da iteracao i=%d: s=%d (prefixo A[0..%d])\n", i, s, i - 1);

        s += A[i];
    }
    return s;
}

int main(void) {
    int A[] = {4, 7, 1, 3};
    printf("soma = %d\n", soma(A, 4));
    return 0;
}


In [ ]:
# Compila e executa: o assert confere o invariante a cada iteração
!gcc -Wall invariante_observado.c -o invariante_observado && ./invariante_observado
!./invariante_observado | grep -q 'soma = 15' && echo OK || echo 'Verifique: esperava soma = 15'


In [ ]:
%%writefile exercicio1.c
#include <stdio.h>
#include <assert.h>

/* Exercício 1: conte quantos elementos de A[0..n-1] são pares.
   TODO: complete o corpo do laço E o assert que confere o invariante:
   "no inicio da iteracao i, cont == quantidade de pares em A[0..i-1]" */
int conta_pares(int A[], int n) {
    int cont = 0;
    for (int i = 0; i < n; i++) {
        /* TODO: recalcule aqui a quantidade de pares do prefixo A[0..i-1]
           e escreva um assert comparando com cont */

        /* TODO: atualize cont */
    }
    return cont;
}

int main(void) {
    int A[] = {4, 7, 2, 3, 8, 8};
    printf("pares = %d\n", conta_pares(A, 6));
    return 0;
}


In [ ]:
# Teste automático do Exercício 1
!gcc -Wall exercicio1.c -o exercicio1 && ./exercicio1
!./exercicio1 | grep -q 'pares = 4' && echo OK || echo 'Verifique sua implementação: esperava pares = 4'


## 2. Busca do máximo

O invariante da busca do máximo é a formalização da frase "`max` é o maior valor **já visto**":

> No início da iteração de índice `i`: `max == max(A[0..i-1])`.

- **Inicialização** (`i = 1`): `max = A[0]` é o máximo do prefixo de tamanho 1. ✓
- **Manutenção**: se `A[i] > max`, o novo `max` é `A[i]`; senão `max` já domina `A[i]`.
  Nos dois casos, `max == max(A[0..i])` ao fim da iteração. ✓
- **Término** (`i = n`): `max == max(A[0..n-1])` — a poscondição. ✓


In [ ]:
%%writefile maximo.c
#include <stdio.h>
#include <assert.h>

int maximo(int A[], int n) {           /* pre: n >= 1 */
    assert(n >= 1);
    int max = A[0];
    for (int i = 1; i < n; i++) {
        /* invariante: max == maior valor de A[0..i-1] */
        int m = A[0];                  /* recalcula o maximo do prefixo */
        for (int k = 1; k < i; k++)
            if (A[k] > m) m = A[k];
        assert(max == m);

        if (A[i] > max)
            max = A[i];
    }
    return max;
}

int main(void) {
    int A[] = {3, 9, 4, 7, 1, 8};
    int B[] = {-5, -2, -9};
    printf("max A = %d\n", maximo(A, 6));
    printf("max B = %d\n", maximo(B, 3));
    return 0;
}


In [ ]:
# Compila e executa — repare no caso B: todos os valores negativos
!gcc -Wall maximo.c -o maximo && ./maximo
!./maximo | grep -q 'max A = 9' && ./maximo | grep -q 'max B = -2' && echo OK || echo 'Verifique: esperava max A = 9 e max B = -2'


### O erro que a inicialização denuncia

Troque mentalmente `int max = A[0];` por `int max = 0;` (e comece o laço em `i = 0`).
A inicialização do invariante falha: antes da primeira iteração, `max` teria que ser o
máximo do prefixo **vazio** — que não existe. E, de fato, para `B = {-5, -2, -9}` a
função devolveria `0`, um valor que nem está no vetor.

Execute a célula abaixo para ver o invariante **flagrar** o erro em tempo de execução.


In [ ]:
%%writefile maximo_errado.c
#include <stdio.h>
#include <assert.h>

int maximo_errado(int A[], int n) {
    int max = 0;                       /* ERRO: inicializacao "no chute" */
    for (int i = 1; i < n; i++) {
        int m = A[0];
        for (int k = 1; k < i; k++)
            if (A[k] > m) m = A[k];
        assert(max == m);              /* invariante: vai falhar em i=1  */

        if (A[i] > max)
            max = A[i];
    }
    return max;
}

int main(void) {
    int B[] = {-5, -2, -9};
    printf("max B = %d\n", maximo_errado(B, 3));
    return 0;
}


In [ ]:
# O assert aborta o programa: o invariante não vale já na primeira iteração
!gcc -Wall maximo_errado.c -o maximo_errado && ./maximo_errado; echo "status de saida: $?"


In [ ]:
%%writefile exercicio2.c
#include <stdio.h>
#include <assert.h>

/* Exercício 2: devolva o INDICE da primeira ocorrencia do maximo de A[0..n-1].
   Invariante sugerido: "no inicio da iteracao i, imax e o indice da primeira
   ocorrencia do maximo de A[0..i-1]".
   TODO: implemente a funcao mantendo esse invariante. Atencao ao criterio
   "primeira ocorrencia": quando A[i] == A[imax], imax NAO deve mudar. */
int indice_maximo(int A[], int n) {
    /* TODO: implemente aqui */
    return -1;
}

int main(void) {
    int A[] = {3, 9, 4, 9, 1};
    int C[] = {7, 7, 7};
    printf("imax A = %d\n", indice_maximo(A, 5));
    printf("imax C = %d\n", indice_maximo(C, 3));
    return 0;
}


In [ ]:
# Teste automático do Exercício 2
!gcc -Wall exercicio2.c -o exercicio2 && ./exercicio2
!./exercicio2 | grep -q 'imax A = 1' && ./exercicio2 | grep -q 'imax C = 0' && echo OK || echo 'Verifique sua implementação: esperava imax A = 1 e imax C = 0'


## 3. Ordenação por inserção

Agora o invariante afirma uma **propriedade estrutural** de um trecho do vetor:

> No início da iteração `i` do laço externo: `A[0..i-1]` está **ordenado** e contém
> **os mesmos elementos** que ocupavam `A[0..i-1]` no vetor original.

A segunda cláusula importa: um laço que escrevesse `A[i] = i` deixaria o vetor
"ordenado" — e errado. Como a inserção apenas **desloca** e **recoloca** valores,
nenhum elemento é criado nem perdido.

No código abaixo, o `assert` confere a cláusula de ordem a cada iteração do laço externo.


In [ ]:
%%writefile insercao.c
#include <stdio.h>
#include <assert.h>

void insercao(int A[], int n) {
    for (int i = 1; i < n; i++) {
        /* invariante: A[0..i-1] ordenado (mesmos elementos do original) */
        for (int k = 0; k < i - 1; k++)
            assert(A[k] <= A[k + 1]);
        printf("inicio de i=%d: prefixo ordenado ate A[%d]\n", i, i - 1);

        int chave = A[i];
        int j = i - 1;
        while (j >= 0 && A[j] > chave) {
            A[j + 1] = A[j];
            j--;
        }
        A[j + 1] = chave;
    }
}

int main(void) {
    int A[] = {5, 2, 4, 6, 1, 3};
    insercao(A, 6);
    for (int k = 0; k < 6; k++)
        printf("%d ", A[k]);
    printf("\n");
    return 0;
}


In [ ]:
# Compila e executa: o rastro mostra o invariante crescendo iteração a iteração
!gcc -Wall insercao.c -o insercao && ./insercao
!./insercao | grep -q '1 2 3 4 5 6' && echo OK || echo 'Verifique: esperava 1 2 3 4 5 6'


In [ ]:
%%writefile exercicio3.c
#include <stdio.h>
#include <assert.h>

/* Exercício 3: ordenação por SELECAO.
   Invariante do laco externo: "no inicio da iteracao i, A[0..i-1] contem os i
   menores elementos do vetor, em ordem".
   TODO: implemente a selecao mantendo esse invariante: a cada iteracao,
   encontre o menor elemento de A[i..n-1] e troque-o com A[i].
   Repare: o invariante da selecao e MAIS FORTE que o da insercao
   (o prefixo ja esta na posicao final). */
void selecao(int A[], int n) {
    /* TODO: implemente aqui */
}

int main(void) {
    int A[] = {5, 2, 4, 6, 1, 3};
    selecao(A, 6);
    for (int k = 0; k < 6; k++)
        printf("%d ", A[k]);
    printf("\n");
    return 0;
}


In [ ]:
# Teste automático do Exercício 3
!gcc -Wall exercicio3.c -o exercicio3 && ./exercicio3
!./exercicio3 | grep -q '1 2 3 4 5 6' && echo OK || echo 'Verifique sua implementação: esperava 1 2 3 4 5 6'


### Para pensar (responda em uma célula de texto)

1. Na ordenação por inserção, por que o invariante diz "os **mesmos** elementos" em vez
   de só "ordenado"? Dê um exemplo de laço que satisfaz a cláusula de ordem e viola a de
   permutação.
2. O invariante da seleção ("os `i` menores, em ordem, nas posições finais") é mais forte
   que o da inserção ("prefixo ordenado"). O que essa diferença implica sobre o trabalho
   que cada iteração precisa fazer?


## Desafio Final

Implemente `int eh_ordenado(int A[], int n)` que devolve `1` se `A[0..n-1]` está em ordem
não decrescente e `0` caso contrário — **e enuncie o invariante** do seu laço em comentário,
no formato visto em aula.

Depois, use-a para escrever `verifica_ordenacao(int orig[], int A[], int n)`, que confere
a poscondição **completa** de um algoritmo de ordenação:

1. `A` está ordenado (`eh_ordenado`);
2. `A` é permutação de `orig` — para vetores de inteiros pequenos, uma forma simples é
   comparar contagens de ocorrência de cada valor.

Rode sua verificação sobre a `insercao` e a `selecao` deste tutorial. Você acabou de
construir um **verificador de poscondição**: ele não substitui a prova, mas automatiza a
conferência em qualquer teste.


In [ ]:
%%writefile desafio.c
#include <stdio.h>

/* Desafio: verificador de poscondição de ordenação */

int eh_ordenado(int A[], int n) {
    /* TODO: implemente e enuncie o invariante em comentario */
    return 0;
}

int verifica_ordenacao(int orig[], int A[], int n) {
    /* TODO: 1) A ordenado; 2) A permutacao de orig */
    return 0;
}

int main(void) {
    /* TODO: copie um vetor, ordene a copia com sua insercao/selecao
       e imprima o veredito de verifica_ordenacao */
    return 0;
}


In [ ]:
# Compile e teste seu desafio
!gcc -Wall desafio.c -o desafio && ./desafio


## Referências

Veja o arquivo `../referencias.bib` para a lista completa.
